# 2. Travel Classification
## Coast-to-Coast Travel Analysis

This notebook:
- Classifies all 32 NFL teams by time zone
- Calculates time zones crossed for each game
- Identifies coast-to-coast travel games (3 time zones)
- Analyzes travel burden by team

**Output**: Schedule data with travel metrics added

---
## Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load schedule data from previous notebook
schedules = pd.read_csv('data_schedules.csv')
print(f"✓ Loaded {len(schedules):,} games")

---
## Team Time Zone Classification

NFL teams distributed across 4 US time zones:
- **Pacific** (0 offset): West Coast teams
- **Mountain** (+1 hour): Denver, Arizona
- **Central** (+2 hours): Midwest and South
- **Eastern** (+3 hours): East Coast teams

In [ ]:
# Team time zone mapping
team_timezones = {
    # Pacific (West Coast) - 5 teams
    'SEA': 'Pacific',
    'SF': 'Pacific',
    'LAC': 'Pacific',
    'LAR': 'Pacific',
    'LV': 'Pacific',  # Las Vegas
    
    # Mountain - 2 teams
    'DEN': 'Mountain',
    'ARI': 'Mountain',
    
    # Central - 10 teams
    'CHI': 'Central',
    'GB': 'Central',
    'MIN': 'Central',
    'DET': 'Central',
    'DAL': 'Central',
    'HOU': 'Central',
    'KC': 'Central',
    'NO': 'Central',
    'TEN': 'Central',
    'IND': 'Central',
    
    # Eastern (East Coast) - 15 teams
    'BUF': 'Eastern',
    'MIA': 'Eastern',
    'NE': 'Eastern',
    'NYJ': 'Eastern',
    'BAL': 'Eastern',
    'CIN': 'Eastern',
    'CLE': 'Eastern',
    'PIT': 'Eastern',
    'ATL': 'Eastern',
    'CAR': 'Eastern',
    'TB': 'Eastern',
    'WAS': 'Eastern',
    'NYG': 'Eastern',
    'PHI': 'Eastern',
    'JAX': 'Eastern'
}

# Time zone offsets (hours from Pacific)
timezone_offset = {
    'Pacific': 0,
    'Mountain': 1,
    'Central': 2,
    'Eastern': 3
}

In [ ]:
# Count teams by time zone
from collections import Counter
tz_counts = Counter(team_timezones.values())

print("Teams by Time Zone:")
for tz in ['Pacific', 'Mountain', 'Central', 'Eastern']:
    print(f"  {tz:10s}: {tz_counts[tz]:2d} teams")

---
## Add Time Zones to Schedule Data

In [ ]:
# Map time zones to schedule
schedules['home_timezone'] = schedules['home_team'].map(team_timezones)
schedules['away_timezone'] = schedules['away_team'].map(team_timezones)

# Check for any unmapped teams
unmapped = schedules[schedules['home_timezone'].isna() | schedules['away_timezone'].isna()]
if len(unmapped) > 0:
    print(f"⚠ Warning: {len(unmapped)} games with unmapped teams")
else:
    print("✓ All teams successfully mapped to time zones")

---
## Calculate Travel Distance

For each game, calculate how many time zones the away team crossed.
- 0 zones = same time zone game
- 1 zone = short travel
- 2 zones = moderate travel
- 3 zones = **coast-to-coast** (our focus)

In [ ]:
def calculate_tz_difference(home_tz, away_tz):
    """
    Calculate number of time zones traveled by away team.
    
    Args:
        home_tz: Home team's time zone
        away_tz: Away team's time zone
    
    Returns:
        int: Number of time zones crossed (absolute value)
    """
    if pd.isna(home_tz) or pd.isna(away_tz):
        return 0
    
    home_offset = timezone_offset.get(home_tz, 0)
    away_offset = timezone_offset.get(away_tz, 0)
    
    return abs(home_offset - away_offset)

# Apply function to calculate travel
schedules['tz_crossed'] = schedules.apply(
    lambda row: calculate_tz_difference(row['home_timezone'], row['away_timezone']),
    axis=1
)

print("✓ Time zones crossed calculated for all games")

---
## Identify Coast-to-Coast Games

In [ ]:
# Flag coast-to-coast travel (3 time zones)
schedules['coast_to_coast'] = schedules['tz_crossed'] == 3

# Summary statistics
print("Travel Distribution:")
print(schedules['tz_crossed'].value_counts().sort_index())
print()
print(f"Coast-to-coast games: {schedules['coast_to_coast'].sum():,}")
print(f"Percentage of all games: {schedules['coast_to_coast'].sum() / len(schedules) * 100:.1f}%")

---
## Analyze Travel Burden by Team

Which teams face the most coast-to-coast travel?

In [ ]:
# Coast-to-coast games as AWAY team (the team that travels)
away_c2c = schedules[schedules['coast_to_coast']].groupby('away_team').size().sort_values(ascending=False)

print("Teams with Most Coast-to-Coast AWAY Games (2021-2024):")
print(away_c2c.head(10))

In [ ]:
# Coast-to-coast games as HOME team (opponents traveling to them)
home_c2c = schedules[schedules['coast_to_coast']].groupby('home_team').size().sort_values(ascending=False)

print("Teams HOSTING Most Coast-to-Coast Games (2021-2024):")
print(home_c2c.head(10))

---
## Visualize Travel Burden

In [ ]:
# Create visualization of travel burden
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Away team travel burden
away_c2c.plot(kind='barh', ax=ax1, color='steelblue')
ax1.set_title('Coast-to-Coast Travel Burden\n(Away Games)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Number of Games', fontsize=12)
ax1.set_ylabel('Team', fontsize=12)

# Home team hosting
home_c2c.plot(kind='barh', ax=ax2, color='coral')
ax2.set_title('Coast-to-Coast Games Hosted\n(Home Games)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Number of Games', fontsize=12)
ax2.set_ylabel('Team', fontsize=12)

plt.tight_layout()
plt.show()

---
## East Coast vs West Coast Disparity

In [ ]:
# Classify teams by coast
west_coast_teams = [team for team, tz in team_timezones.items() if tz == 'Pacific']
east_coast_teams = [team for team, tz in team_timezones.items() if tz == 'Eastern']

# Calculate average travel burden
west_avg = away_c2c[away_c2c.index.isin(west_coast_teams)].mean()
east_avg = away_c2c[away_c2c.index.isin(east_coast_teams)].mean()

print("Average Coast-to-Coast Away Games per Team (2021-2024):")
print(f"  West Coast teams ({len(west_coast_teams)} teams): {west_avg:.1f} games")
print(f"  East Coast teams ({len(east_coast_teams)} teams): {east_avg:.1f} games")
print(f"\n  Disparity: West Coast teams face {west_avg - east_avg:.1f} more games on average")

---
## Save Enhanced Schedule Data

In [ ]:
# Save schedule with travel metrics
schedules.to_csv('data_schedules_with_travel.csv', index=False)

print("✓ Enhanced schedule data saved")
print("  - data_schedules_with_travel.csv")
print("\nNew columns added:")
print("  - home_timezone")
print("  - away_timezone")
print("  - tz_crossed")
print("  - coast_to_coast")

---
## Summary

**Key Findings:**
- Coast-to-coast games identified across 4 seasons
- West Coast teams face significantly more travel burden
- Clear disparity in workload between coasts

**Next Steps:**
- Run `03_performance_impact.ipynb` to analyze how travel affects performance
- Measure win rates, scoring, and other metrics